# 02 — Limpeza dos dados do ClinicalTrials.gov
Este notebook transforma os dados brutos (extração de 12/10/2026) em tabelas limpas para análise.


## 1. Ler os dados brutos

In [34]:
import json
import pandas as pd

In [35]:
with open("../data/raw/estudos_obesidade_2026-10-12.json", encoding="utf-8") as ficheiro:
    estudos_brutos = json.load(ficheiro)

In [36]:
len(estudos_brutos)

8431

In [37]:
print(json.dumps(estudos_brutos[0], indent=2, ensure_ascii=False))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT06151145",
      "briefTitle": "Stepped Care for Weight Loss Maintenance"
    },
    "statusModule": {
      "overallStatus": "ACTIVE_NOT_RECRUITING",
      "startDateStruct": {
        "date": "2024-02-01",
        "type": "ACTUAL"
      },
      "primaryCompletionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      },
      "completionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      }
    },
    "sponsorCollaboratorsModule": {
      "leadSponsor": {
        "class": "OTHER"
      }
    },
    "designModule": {
      "studyType": "INTERVENTIONAL",
      "phases": [
        "NA"
      ],
      "enrollmentInfo": {
        "count": 192
      }
    },
    "armsInterventionsModule": {
      "interventions": [
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        }
      

In [38]:
print(json.dumps(estudos_brutos[0]["protocolSection"]["designModule"], indent=2))

{
  "studyType": "INTERVENTIONAL",
  "phases": [
    "NA"
  ],
  "enrollmentInfo": {
    "count": 192
  }
}


In [39]:
for estudo in estudos_brutos:
    status = estudo["protocolSection"]["statusModule"]
    if status["overallStatus"] == "TERMINATED":
        print(json.dumps(status, indent=2))
        break

{
  "overallStatus": "TERMINATED",
  "whyStopped": "difficulty in recruiting",
  "startDateStruct": {
    "date": "2013-12",
    "type": "ACTUAL"
  },
  "primaryCompletionDateStruct": {
    "date": "2022-01",
    "type": "ACTUAL"
  },
  "completionDateStruct": {
    "date": "2022-06",
    "type": "ACTUAL"
  }
}


## 2. Construir a tabela estudos

In [59]:
linhas = []
for estudo in estudos_brutos:
   
    protocolo = estudo.get("protocolSection", {})

    
    ident = protocolo.get("identificationModule", {})
    status =protocolo.get("statusModule",{})
    design = protocolo.get("designModule",{})
    patrocinio = protocolo.get("sponsorCollaboratorsModule",{})

    
    inicio = status.get("startDateStruct", {})
    fim_primaria = status.get("primaryCompletionDateStruct",{})
    fim = status.get("completionDateStruct",{})
    inscricao = design.get("enrollmentInfo",{})
    patrocinador = patrocinio.get("leadSponsor",{})

   
    fases = design.get("phases",[])

    linhas.append({
        "nct_id": ident.get("nctId"),
        "titulo": ident.get("briefTitle"),
        "tipo_estudo": design.get("studyType"),
        "estado": status.get("overallStatus"),
        "motivo_paragem": status.get("whyStopped"),
        "data_inicio": inicio.get("date"),
        "tipo_data_inicio": inicio.get("type"),
        "data_fim_primaria": fim_primaria.get("date"),
        "tipo_data_fim_primaria": fim_primaria.get("type"),
        "data_fim": fim.get("date"),
        "tipo_data_fim": fim.get("type"),
        "participantes": inscricao.get("count"),
        "fase": "/".join(fases) if fases else None,
        "classe_patrocinador": patrocinador.get("class"),
    })

df_estudos = pd.DataFrame(linhas)

In [41]:
df_estudos.shape

(8431, 14)

In [42]:
df_estudos.head()

,nct_id,titulo,tipo_estudo,estado,motivo_paragem,data_inicio,tipo_data_inicio,data_fim_primaria,tipo_data_fim_primaria,data_fim,tipo_data_fim,participantes,fase,classe_patrocinador
0,NCT06151145,Stepped Care for Weight Loss Maintenance,INTERVENTIONAL,ACTIVE_NOT_RECRUITING,NaN,2024-02-01,ACTUAL,2028-06,ESTIMATED,2028-06,ESTIMATED,192.0,NA,OTHER
1,NCT02185872,CrossFit Exercise to Improve Glucose Control f...,INTERVENTIONAL,COMPLETED,NaN,2012-02,NaN,2012-05,ACTUAL,2012-05,ACTUAL,23.0,NA,OTHER
2,NCT03917901,Computer-Delivered Intervention for Individual...,INTERVENTIONAL,COMPLETED,NaN,2019-09-16,ACTUAL,2021-03-06,ACTUAL,2021-03-06,ACTUAL,131.0,NA,OTHER
3,NCT01049698,Improving Muscle for Functional Independence T...,INTERVENTIONAL,COMPLETED,NaN,2010-02,NaN,2014-05,ACTUAL,2014-05,ACTUAL,126.0,NA,OTHER
4,NCT06989177,Probiotic Intervention on Body Weight,INTERVENTIONAL,RECRUITING,NaN,2025-06-10,ACTUAL,2026-12-30,ESTIMATED,2026-12-31,ESTIMATED,140.0,NA,OTHER_GOV


In [43]:
df_estudos.isna().sum()

nct_id                       0
titulo                       0
tipo_estudo                  0
estado                       0
motivo_paragem            7935
data_inicio                  0
tipo_data_inicio          2062
data_fim_primaria            3
tipo_data_fim_primaria       4
data_fim                    54
tipo_data_fim               57
participantes                2
fase                         3
classe_patrocinador          0
dtype: int64

In [44]:
parados = df_estudos["estado"].isin(["TERMINATED", "SUSPENDED", "WITHDRAWN"])
df_estudos[parados]["motivo_paragem"].isna().sum()

np.int64(40)

In [45]:
df_estudos["nct_id"].is_unique

True

In [46]:
df_estudos["tipo_estudo"].value_counts(dropna=False)

tipo_estudo
INTERVENTIONAL    8431
Name: count, dtype: int64

In [47]:
sem_tipo = df_estudos["tipo_data_inicio"].isna()
df_estudos[sem_tipo]["data_inicio"].str[:4].value_counts().sort_index()

data_inicio
2010    325
2011    317
2012    353
2013    323
2014    280
2015    263
2016    193
2017      8
Name: count, dtype: int64

In [48]:
df_estudos["tipo_data_inicio"].isna().mean()

np.float64(0.24457359743802634)

In [49]:
parados.sum()

np.int64(536)

In [50]:
df_estudos[~parados]["motivo_paragem"].notna().sum()

np.int64(0)

In [51]:
(df_estudos["data_fim"].notna() & df_estudos["tipo_data_fim"].isna()).sum()

np.int64(3)

In [56]:
df_estudos["participantes"]= df_estudos["participantes"].astype("Int64")

In [57]:
df_estudos.info()

<class 'pandas.DataFrame'>
RangeIndex: 8431 entries, 0 to 8430
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   nct_id                  8431 non-null   str  
 1   titulo                  8431 non-null   str  
 2   tipo_estudo             8431 non-null   str  
 3   estado                  8431 non-null   str  
 4   motivo_paragem          496 non-null    str  
 5   data_inicio             8431 non-null   str  
 6   tipo_data_inicio        6369 non-null   str  
 7   data_fim_primaria       8428 non-null   str  
 8   tipo_data_fim_primaria  8427 non-null   str  
 9   data_fim                8377 non-null   str  
 10  tipo_data_fim           8374 non-null   str  
 11  participantes           8429 non-null   Int64
 12  fase                    8428 non-null   str  
 13  classe_patrocinador     8431 non-null   str  
dtypes: Int64(1), str(13)
memory usage: 930.5 KB


### Observações da inspeção

- **Dimensão:** a tabela tem 8 431 estudos e 14 colunas.
- **Validações:** o `nct_id` é único (não há estudos repetidos) e os 8 431 estudos são todos INTERVENTIONAL, o que confirma o filtro da extração.
- **Estudos parados sem motivo:** dos 536 estudos parados, 40 não indicam o motivo. Não há estudos não parados com motivo.
- **Tipo da data de início:** 24,5 % dos estudos não têm tipo (real ou estimada). Todos começaram entre 2010 e 2017; a partir de 2018 não há nenhum caso, o que sugere uma mudança na forma como os dados eram recolhidos.
- **Data de fim:** 54 estudos não têm data de fim. Outros 3 têm a data, mas não indicam se é real ou estimada.
- **Data de fim primária:** 3 estudos não têm esta data. Outro 1 tem a data, mas não indica se é real ou estimada.
- **Participantes e fase:** 2 estudos não indicam o número de participantes e 3 não indicam a fase.